In [0]:
%sql

SELECT ai_query(
    'system.ai.gpt-oss-20b',
    'Reply with exactly the word OK.'
) AS test;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW ai_source_run AS

SELECT run_id
FROM workspace.synthetic_profile.profile_runs

WHERE profile_type = 'SOURCE'
  AND status = 'COMPLETED'

QUALIFY ROW_NUMBER() OVER (
    ORDER BY started_at DESC
) = 1;

In [0]:
%sql

SELECT *
FROM ai_source_run;

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.synthetic_profile.ai_column_context
USING DELTA
AS

WITH source_run AS (

    SELECT run_id
    FROM ai_source_run
),

category_ranked AS (

    SELECT
        c.logical_table_name,
        c.column_name,
        c.category_value,
        c.proportion,

        ROW_NUMBER() OVER (
            PARTITION BY
                c.logical_table_name,
                c.column_name
            ORDER BY c.proportion DESC
        ) AS rn

    FROM workspace.synthetic_profile.category_profile c

    JOIN source_run s
        ON c.run_id = s.run_id
),

categories AS (

    SELECT
        logical_table_name,
        column_name,

        COLLECT_LIST(
            NAMED_STRUCT(
                'value', category_value,
                'proportion', proportion
            )
        ) AS top_categories

    FROM category_ranked

    WHERE rn <= 10

    GROUP BY
        logical_table_name,
        column_name
)

SELECT
    i.logical_table_name,
    i.column_name,
    i.data_type,
    i.column_role,

    q.null_pct,
    q.distinct_count,

    n.mean_value,
    n.p25,
    n.p50,
    n.p75,
    n.p95,
    n.stddev_value,

    c.top_categories

FROM workspace.synthetic_profile.column_inventory i

CROSS JOIN source_run s

LEFT JOIN workspace.synthetic_profile.column_quality q
    ON q.run_id = s.run_id
   AND q.logical_table_name = i.logical_table_name
   AND q.column_name = i.column_name

LEFT JOIN workspace.synthetic_profile.numeric_profile n
    ON n.run_id = s.run_id
   AND n.logical_table_name = i.logical_table_name
   AND n.column_name = i.column_name

LEFT JOIN categories c
    ON c.logical_table_name = i.logical_table_name
   AND c.column_name = i.column_name

WHERE i.logical_table_name IN (
    'TKQ_ACCOUNT',
    'TKQ_ACCOUNT_BALANCE'
);

What llm sees:

In [0]:
%sql

SELECT *
FROM workspace.synthetic_profile.ai_column_context
ORDER BY logical_table_name, column_name;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW ai_account_rule_input AS

SELECT
    'ACCOUNT_CORE' AS scope_name,

    CONCAT_WS(
        '\n',
        COLLECT_LIST(
            TO_JSON(
                NAMED_STRUCT(
                    'table', logical_table_name,
                    'column', column_name,
                    'type', data_type,
                    'role', column_role,

                    'null_pct', null_pct,
                    'distinct_count', distinct_count,

                    'mean', mean_value,
                    'p25', p25,
                    'p50', p50,
                    'p75', p75,
                    'p95', p95,

                    'top_categories', top_categories
                )
            )
        )
    ) AS profile_summary

FROM workspace.synthetic_profile.ai_column_context;

In [0]:
%sql

SELECT *
FROM ai_account_rule_input;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.synthetic_profile.ai_rule_proposals (

    proposal_id STRING,
    created_at TIMESTAMP,
    source_run_id STRING,

    target_table STRING,
    target_column STRING,

    rule_type STRING,

    parent_columns ARRAY<STRING>,

    confidence STRING,
    evidence_type STRING,

    statistical_evidence STRING,
    rationale STRING,
    proposed_rule STRING,

    requires_sme_review BOOLEAN,
    approval_status STRING

)
USING DELTA;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW ai_raw_rule_proposals AS

SELECT
    scope_name,

    ai_query(

        'system.ai.gpt-oss-20b',

        CONCAT(

'You are assisting with a banking synthetic-data generation system.

You receive ONLY aggregate statistical profiles. You do not have row-level records.

Your task is to propose candidate generation dependencies and business rules.

A good candidate rule must:
1. improve logical or statistical consistency;
2. identify fields that should not be generated independently;
3. distinguish statistical evidence from domain inference;
4. avoid inventing relationships unsupported by the supplied evidence;
5. mark domain-inferred rules for SME review;
6. prefer deterministic DERIVED rules where a field is clearly derived from others;
7. use CONDITIONAL when a target distribution should depend on another variable;
8. use CORRELATED only when a numeric association should be preserved but no deterministic formula is supported;
9. use NO_RULE when independent generation is acceptable or evidence is insufficient.

Do NOT claim causation from correlation.

For each proposed rule identify:
- target table
- target column
- rule type
- parent columns
- confidence: HIGH, MEDIUM, or LOW
- evidence type: STATISTICAL, DOMAIN_INFERENCE, or STATISTICAL_AND_DOMAIN
- statistical evidence
- rationale
- proposed generation rule
- whether SME review is required

Here are the profiler statistics:

',
            profile_summary
        ),

        responseFormat => '{
          "type": "json_schema",
          "json_schema": {
            "name": "banking_generation_rules",
            "schema": {
              "type": "object",
              "properties": {
                "rules": {
                  "type": "array",
                  "items": {
                    "type": "object",
                    "properties": {
                      "target_table": {
                        "type": "string"
                      },
                      "target_column": {
                        "type": "string"
                      },
                      "rule_type": {
                        "type": "string"
                      },
                      "parent_columns": {
                        "type": "array",
                        "items": {
                          "type": "string"
                        }
                      },
                      "confidence": {
                        "type": "string"
                      },
                      "evidence_type": {
                        "type": "string"
                      },
                      "statistical_evidence": {
                        "type": "string"
                      },
                      "rationale": {
                        "type": "string"
                      },
                      "proposed_rule": {
                        "type": "string"
                      },
                      "requires_sme_review": {
                        "type": "boolean"
                      }
                    },
                    "required": [
                      "target_table",
                      "target_column",
                      "rule_type",
                      "parent_columns",
                      "confidence",
                      "evidence_type",
                      "statistical_evidence",
                      "rationale",
                      "proposed_rule",
                      "requires_sme_review"
                    ],
                    "additionalProperties": false
                  }
                }
              },
              "required": [
                "rules"
              ],
              "additionalProperties": false
            },
            "strict": true
          }
        }'

    ) AS response

FROM ai_account_rule_input;

In [0]:
%sql

SELECT
    scope_name,
    response

FROM ai_raw_rule_proposals;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW ai_parsed_rule_proposals AS

SELECT
    scope_name,

    FROM_JSON(
        response,

        'STRUCT<
            rules:ARRAY<
                STRUCT<
                    target_table:STRING,
                    target_column:STRING,
                    rule_type:STRING,
                    parent_columns:ARRAY<STRING>,
                    confidence:STRING,
                    evidence_type:STRING,
                    statistical_evidence:STRING,
                    rationale:STRING,
                    proposed_rule:STRING,
                    requires_sme_review:BOOLEAN
                >
            >
        >'
    ) AS parsed_response

FROM ai_raw_rule_proposals;

In [0]:
%sql

INSERT INTO workspace.synthetic_profile.ai_rule_proposals

SELECT
    uuid() AS proposal_id,

    current_timestamp() AS created_at,

    (
        SELECT run_id
        FROM ai_source_run
        LIMIT 1
    ) AS source_run_id,

    rule.target_table,
    rule.target_column,
    rule.rule_type,
    rule.parent_columns,
    rule.confidence,
    rule.evidence_type,
    rule.statistical_evidence,
    rule.rationale,
    rule.proposed_rule,
    rule.requires_sme_review,

    'PENDING' AS approval_status

FROM ai_parsed_rule_proposals

LATERAL VIEW EXPLODE(
    parsed_response.rules
) exploded AS rule;

In [0]:
%sql

SELECT
    target_table,
    target_column,
    rule_type,
    parent_columns,
    confidence,
    evidence_type,
    statistical_evidence,
    proposed_rule,
    rationale,
    requires_sme_review,
    approval_status

FROM workspace.synthetic_profile.ai_rule_proposals

ORDER BY
    target_table,
    target_column;

In [0]:
%sql

UPDATE workspace.synthetic_profile.ai_rule_proposals
SET approval_status = 'APPROVED'
WHERE approval_status <> 'APPROVED';